# 02 · Baseline eval — model A (base student, zero-shot) and T (teacher/generator, optional)
**Purpose:** serve each model with SGLang, run the 500 TEST items through AuditKIT, save metrics.
**Inputs:** `artifacts/data/test.jsonl`. **Outputs:** `results/eval_{A,T}.json`, `results/predictions_{A,T}.jsonl`.
**Runtime (T4):** ~5 min for A, ~10 min for T (server start + 500 generations each).

In [ ]:
import os, sys, json
if not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/ps8/repo/src")
import ps8_common as C
C.ensure_env()
cfg = C.load_config(); FORCE = False
GPU, P = C.gpu_profile(cfg)
RUN_T = True   # optional teacher reference; first thing to cut if short on time
print("profile", GPU)

In [ ]:
# Model A: base student, zero-shot.
if FORCE or not (C.RESULTS / "eval_A.json").exists():
    C.start_server(cfg["student"], "A", P["sglang_args"], mem_fraction=P["mem_fraction"])
    C.evaluate_on_test("A", "A", cfg)
    C.stop_server()
A = json.load(open(C.RESULTS / "eval_A.json"))

In [ ]:
# Model T: the generator/teacher itself, zero-shot (reference ceiling for distillation).
# Needs the generator chosen in notebook 03 (data/generator_choice.json); skipped until then.
choice_f = C.DATA / "generator_choice.json"
if RUN_T and choice_f.exists() and (FORCE or not (C.RESULTS / "eval_T.json").exists()):
    GEN = json.load(open(choice_f))["generator"]
    C.start_server(GEN, "T", P["sglang_args"] + (P["generator_args"] if "AWQ" in GEN else []), mem_fraction=P["mem_fraction"])
    C.evaluate_on_test("T", "T", cfg)
    C.stop_server()

In [ ]:
for m in ("A", "T"):
    f = C.RESULTS / f"eval_{m}.json"
    if f.exists():
        r = json.load(open(f))
        print(f"{m}: acc={r['accuracy']:.3f} CI{[round(x,3) for x in r['accuracy_ci95']]} macroF1={r['macro_f1']:.3f} "
              f"CI{[round(x,3) for x in r['macro_f1_ci95']]} fmt_fail={r['format_failure_rate']:.3f} per-class={r['f1_per_class']}")
print("Saved:", sorted(p.name for p in C.RESULTS.glob("eval_*.json")))